# Make PanDerm CSVs
Converts `dataset_manifest.csv` into 5 fold-specific CSVs in the format PanDerm's `run_class_finetuning.py` expects:

```
image, label, split
```

For each fold, that fold is the **test** set, the next fold is **val**, and the rest are **train**.

**Run after** `prepare_data.ipynb` — requires `dataset_manifest.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

In [ ]:
from pathlib import Path

# ── Edit these paths to match your Drive layout ────────────────────────────────
DRIVE_ROOT    = Path('/content/drive/MyDrive/PanDerm_Scripts')
OUTPUT_DIR    = DRIVE_ROOT / 'results'          # contains dataset_manifest.csv
CSV_OUT_DIR   = DRIVE_ROOT / 'cross-fold-csv'   # where fold CSVs will be saved
USE_SEGMENTED = True                            # True = use segmented_cache/ paths
SEGMENTED_DIR = DRIVE_ROOT / 'segmented_cache'
DATA_ROOT     = DRIVE_ROOT / 'dermoscopy'
N_FOLDS       = 5
# ───────────────────────────────────────────────────────────────────────────────

CSV_OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Config ready')
print(f'  Manifest       : {OUTPUT_DIR}/dataset_manifest.csv')
print(f'  Output CSVs    : {CSV_OUT_DIR}')
print(f'  Use segmented  : {USE_SEGMENTED}')

## Cell 3 — Load manifest

In [ ]:
import pandas as pd

manifest_csv = OUTPUT_DIR / 'dataset_manifest.csv'
if not manifest_csv.exists():
    raise FileNotFoundError(f'Manifest not found: {manifest_csv}\nRun prepare_data.ipynb first.')

df = pd.read_csv(manifest_csv)
print(f'Loaded manifest: {len(df)} images, {df["patient_id"].nunique()} patients')
print(df['fold'].value_counts().sort_index())

## Cell 4 — Generate one CSV per fold

In [ ]:
for test_fold in range(N_FOLDS):
    val_fold = (test_fold + 1) % N_FOLDS   # next fold is val

    def fold_to_split(fold):
        if fold == test_fold:  return 'test'
        if fold == val_fold:   return 'val'
        return 'train'

    fold_df = df.copy()

    # Remap image paths to segmented_cache if requested
    if USE_SEGMENTED and 'segmented_path' in fold_df.columns:
        fold_df['image'] = fold_df['segmented_path']
    else:
        fold_df['image'] = fold_df['image_path']

    fold_df['split'] = fold_df['fold'].apply(fold_to_split)

    panderm_df = fold_df[['image', 'label', 'split']]

    out_path = CSV_OUT_DIR / f'panderm_finetuning_fold{test_fold}.csv'
    panderm_df.to_csv(out_path, index=False)

    counts = panderm_df['split'].value_counts()
    print(f'Fold {test_fold} (test) | val=fold{val_fold} | '
          f'train={counts.get("train",0)}  val={counts.get("val",0)}  test={counts.get("test",0)} | '
          f'Saved: {out_path.name}')

print(f'\nAll {N_FOLDS} CSVs saved to: {CSV_OUT_DIR}')

## Cell 5 — Verify one CSV looks correct

In [ ]:
sample = pd.read_csv(CSV_OUT_DIR / 'panderm_finetuning_fold0.csv')
print('Sample rows from fold0 CSV:')
print(sample.head(5).to_string())
print(f'\nColumns : {list(sample.columns)}')
print(f'Split counts:\n{sample["split"].value_counts()}')